In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
import os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import matplotlib.pyplot as plt
from sklearn.model_selection import RandomizedSearchCV
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score,confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit
from xgboost.sklearn import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from scipy.special import entr
from sklearn import preprocessing
import matplotlib.pyplot as plt
import math
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score,confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from IPython.display import display
#import itertools
#from itables import init_notebook_mode
import random
#init_notebook_mode(all_interactive=True)


In [2]:
df = pd.read_csv('my_dict.csv')
user_labels = df['label'].unique()

In [26]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np
import pandas as pd
from sklearn.manifold import Isomap

def run_random_forest(data, user_labels, isomap_applied=False):
    all_y_test = []
    all_y_pred = []

    for target_user in user_labels:
        user_data = data[data['label'] == target_user]
        impostor_data = data[data['label'] != target_user]

        min_samples = min(user_data.shape[0], impostor_data.shape[0])

        balanced_user_data = user_data.sample(min_samples)
        balanced_impostor_data = impostor_data.sample(min_samples)

        balanced_data = pd.concat([balanced_user_data, balanced_impostor_data]).sample(frac=1).reset_index(drop=True)

        balanced_data['label'] = np.where(balanced_data['label'] == target_user, 0, 1)

        X = balanced_data.drop(['label'], axis=1)

        isomap = Isomap(n_neighbors=5, n_components=45)
        print(len(X))
        if len(X) < 100:
            continue
        data_transformed = isomap.fit_transform(X)
        tm = pd.DataFrame()

        # Loop to add the transformed data back to the dataframe as new columns
        for i in range(30):
            tm[f'isomap_{i+1}'] = data_transformed[:, i]

        y = balanced_data['label']

        X_train, X_test, y_train, y_test = train_test_split(tm, y, test_size=0.2, random_state=42)

        # Train the Random Forest classifier on the transformed features
        rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
        rf_classifier.fit(X_train, y_train)

        y_pred = rf_classifier.predict(X_test)

        all_y_test.extend(y_test)
        all_y_pred.extend(y_pred)

    print(f"{'With' if isomap_applied else 'Without'} Isomap:")
    print(classification_report(all_y_test, all_y_pred))
    print()


In [27]:
from sklearn.impute import SimpleImputer



imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
df.iloc[:, :] = imputer.fit_transform(df.replace([np.inf, -np.inf], np.nan))


run_random_forest(df, user_labels)

5358
5824
7332
1400
5066
5712
1416
600
7068
20
1468
4922
10544
9474
934
402
1052
6922
926
Without Isomap:
              precision    recall  f1-score   support

           0       0.63      0.66      0.64      7605
           1       0.65      0.61      0.63      7687

    accuracy                           0.64     15292
   macro avg       0.64      0.64      0.64     15292
weighted avg       0.64      0.64      0.64     15292




In [8]:
len(df.columns)

55

In [10]:
data = df
# Rename label column to user
#data = data.rename(columns={'label': 'user'})
from sklearn.impute import SimpleImputer
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA
# Preprocess your data: replace missing values and infinite values with the mean of the column
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
data.iloc[:, :] = imputer.fit_transform(data.replace([np.inf, -np.inf], np.nan))


# run_random_forest(data, user_labels)


def apply_pca_per_user(data, user_labels, n_components=10):
    transformed_data_list = []

    for target_user in user_labels:
        user_data = data[data['label'] == target_user]

        pca = PCA(n_components=n_components)
        user_transformed_data = pca.fit_transform(user_data.drop(['label'], axis=1))
        user_transformed_data = pd.DataFrame(user_transformed_data)
        #user_transformed_data['user'] = user_data['user'].values
        user_transformed_data['label'] = user_data['label'].values

        transformed_data_list.append(user_transformed_data)

    return pd.concat(transformed_data_list).reset_index(drop=True)



# Run the Random Forest model without dimensionality reduction
# run_random_forest(data, user_labels)

# # Apply PCA for dimensionality reduction for each user
data_transformed = apply_pca_per_user(data, user_labels, n_components=10)

# Run the Random Forest model with PCA-transformed data
run_random_forest(data_transformed, user_labels, isomap_applied=True)
#user_labels

With Isomap:
              precision    recall  f1-score   support

           0       0.83      0.86      0.85      7726
           1       0.86      0.82      0.84      7570

    accuracy                           0.84     15296
   macro avg       0.84      0.84      0.84     15296
weighted avg       0.84      0.84      0.84     15296




MemoryError: Unable to allocate 10.9 GiB for an array with shape (38220, 38220) and data type float64